In [98]:
import pandas as pd

import asyncio
import httpx
from datetime import datetime, time, timedelta
import numpy as np

In [101]:
async def extraer_datos(endpoint, fecha_ini, fecha_fin):
    url = f"http://10.1.0.103:9080/Utea/{endpoint}"
    params = {
        "pStrFecIni": fecha_ini,
        "pStrFecFin": fecha_fin,
    }
    timeout = httpx.Timeout(10.0)
    now = datetime.now()
    formatted_now = now.strftime('%d/%m/%Y %H:%M:%S')
    try:
        async with httpx.AsyncClient(timeout=timeout) as client:
            response = await client.get(url, params=params)
            if response.status_code == 200:
                print("✅ " + formatted_now + f" {endpoint}: GET exitoso")
                return response.json()
            else:
                print("⚠️ " + formatted_now + f" {endpoint}: Error en obtener respuesta")
                return {}
    except httpx.RequestError as e:
        print("❌ " + formatted_now + f" {endpoint}: Error en conexion.")
        return {}

def calcular_horas_trabajadas(fecha_hora_actual=None):
    # Si no se pasa una hora, usamos la del sistema en este momento
    if fecha_hora_actual is None:
        fecha_hora_actual = datetime.now()
    # Definimos que el día laboral empieza a las 6:00 AM
    HORA_INICIO_FABRIL = 6
    # PASO 1: Determinar cuándo empezó el turno actual
    if fecha_hora_actual.hour >= HORA_INICIO_FABRIL:
        # Si son más de las 6:00 AM, el turno empezó HOY a las 6:00 AM
        inicio_turno = fecha_hora_actual.replace(hour=HORA_INICIO_FABRIL, minute=0, second=0, microsecond=0)
    else:
        # Si son entre las 00:00 y las 05:59 AM, el turno empezó AYER a las 6:00 AM
        ayer = fecha_hora_actual - timedelta(days=1)
        inicio_turno = ayer.replace(hour=HORA_INICIO_FABRIL, minute=0, second=0, microsecond=0)
    # PASO 2: Calcular la diferencia de tiempo
    diferencia = fecha_hora_actual - inicio_turno
    # Convertimos la diferencia total a horas (incluyendo minutos como decimales)
    horas_transcurridas = diferencia.total_seconds() / 3600
    return horas_transcurridas

async def get_datos_balanza():
    hoy = datetime.now().strftime('%Y-%m-%d')
    ayer = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
    TrafCamBalanza = await extraer_datos("TrafCamBalanza", ayer, hoy)
    df_trafcambalanza = pd.DataFrame(TrafCamBalanza)
    # convertir "" o " " a null
    df_trafcambalanza['canero'] = df_trafcambalanza['canero'].replace(r'^\s*$', np.nan, regex=True)
    # eliminar registro null
    df_trafcambalanza.dropna(subset=['canero'], inplace=True)
    # converitr columan canero (codigo) a entero
    df_trafcambalanza['canero'] = df_trafcambalanza['canero'].astype(int)
    
    datos_campodulce = df_trafcambalanza[df_trafcambalanza['canero'] == 41594].copy()

    # 1. Limpiar y combinar las columnas
    # Extraemos solo la parte de la hora (HH:MM:SS) de 'startTime' ya que trae un prefijo '0001-01-01T'
    datos_campodulce['time_clean'] = datos_campodulce['endTime'].str.split('T').str[-1]
    # Creamos una columna datetime uniendo la fecha y la hora limpia
    datos_campodulce['fecha_hora'] = pd.to_datetime(datos_campodulce['endDate'] + ' ' + datos_campodulce['time_clean'])
    # 2. Definir el rango del "Día Fabril" actual
    # Obtenemos la fecha de hoy (año-mes-día)
    hoy = datetime.now().date()
    # El inicio del día fabril es hoy a las 6:00 AM
    inicio_fabril = datetime.combine(hoy, time(6, 0, 0))
    # El fin del día fabril es mañana a las 6:00 AM
    fin_fabril = inicio_fabril + pd.Timedelta(days=1)
    # 3. Filtrar el DataFrame
    df_filtrado = datos_campodulce[(datos_campodulce['fecha_hora'] >= inicio_fabril) & (datos_campodulce['fecha_hora'] < fin_fabril)]
    # (Opcional) Borramos la columna temporal de hora limpia si no la necesitas
    df_filtrado = df_filtrado.drop(columns=['time_clean'])
    return df_filtrado

async def carlcular_metricas_mensajes():
    df_trafcambalanza = await get_datos_balanza()
    if len(df_trafcambalanza) == 0:
        return "Sin datos."
    ahora = datetime.now()
    hora_formateada = ahora.strftime("%I:%M %p").lower()

    meta_dia = 3000.00
    avance_hr = meta_dia / 24
    horas_transcurridas = calcular_horas_trabajadas()
    avance_esperado = horas_transcurridas * avance_hr
    avance_actual = df_trafcambalanza['netWeight'].sum()/1000.00
    viajes_recibidos = len(df_trafcambalanza[df_trafcambalanza['netWeight'] > 0])
    avance_porcen = (avance_actual / meta_dia) * 100
    tn_faltantes = meta_dia - avance_actual
    # ritmo
    ritmo_actual = avance_actual / horas_transcurridas
    ritmo_requerido = avance_esperado / horas_transcurridas
    brecha = ritmo_actual - ritmo_requerido
    proyeccion_ciere = ((24 - horas_transcurridas) * ritmo_actual) + avance_actual
    deficit_estimado = proyeccion_ciere - meta_dia
    # necesita
    hr_restantes = (24 - horas_transcurridas)
    ritmo_requerido = tn_faltantes / hr_restantes
    ritmo_viajes_requedito = (tn_faltantes/hr_restantes) / 60
    
    mensaje = f'''*REPORTE DE SUMINISTRO CAÑA F1*

⏱️ Hora: {hora_formateada}
🎯 Meta Día: {meta_dia}tn
🏁 Avan. esperado: {round(avance_esperado,2)}tn
🏃 Avan. actual: {round(avance_actual,2)}tn
🚚 Viajes Recibidos: {round(viajes_recibidos,0)}
📈 Avance: {round(avance_porcen,0)}%
📉 Faltante: {round(tn_faltantes,0)}tn
─────────────────────
⚡ RITMO & PREVISIÓN
─────────────────────
🔄 Ritmo actual: ~{round(ritmo_actual,0)}t/h
📐 Ritmo requerido: ~{round(ritmo_requerido,0)}t/h
⚠️ Brecha: {round(brecha,0)}t/h
🔮 Proyección cierre (ritmo actual): ~{round(proyeccion_ciere,0)}tn
📌 Déficit estimado al cierre: ~{round(deficit_estimado,0)}tn
─────────────────────
✅ ¿QUÉ SE NECESITA?
─────────────────────
▶️ Horas útiles restantes: {round(hr_restantes,2)}h
▶️ Ritmo mínimo requerido: ~{round(ritmo_requerido,0)}t/h
▶️ Con 60 tn/viaje 
→ sostener {round(ritmo_viajes_requedito,0)}viajes/h
'''
    return mensaje

In [102]:
df = await carlcular_metricas_mensajes()
print(df)

✅ 23/05/2026 21:25:19 TrafCamBalanza: GET exitoso
*REPORTE DE SUMINISTRO CAÑA F1*

⏱️ Hora: 09:25 pm
🎯 Meta Día: 3000.0tn
🏁 Avan. esperado: 1927.77tn
🏃 Avan. actual: 1521.49tn
🚚 Viajes Recibidos: 30
📈 Avance: 51.0%
📉 Faltante: 1479.0tn
─────────────────────
⚡ RITMO & PREVISIÓN
─────────────────────
🔄 Ritmo actual: ~99.0t/h
📐 Ritmo requerido: ~172.0t/h
⚠️ Brecha: -26.0t/h
🔮 Proyección cierre (ritmo actual): ~2368.0tn
📌 Déficit estimado al cierre: ~-632.0tn
─────────────────────
✅ ¿QUÉ SE NECESITA?
─────────────────────
▶️ Horas útiles restantes: 8.58h
▶️ Ritmo mínimo requerido: ~172.0t/h
▶️ Con 60 tn/viaje 
→ sostener 3.0viajes/h



In [187]:
datos = await carlcular_metricas_mensajes()

✅ 22/05/2026 16:37:18 TrafCamBalanza: GET exitoso


In [189]:
print(datos)

📊 *REPORTE DE SUMINISTRO CAÑA F1*

⏱️ Hora: 04:37 pm
🎯 Meta Día: 3000tn
🏁 Avan. esperado: 1327.72tn
🏃 Avan. actual: 0.0tn
🚚 Viajes Recibidos: 13
📈 Avance: 0.0%
📉 Faltante: 3000.0tn
─────────────────────
⚡ RITMO & PREVISIÓN
─────────────────────
🔄 Ritmo actual: ~0.0t/h
📐 Ritmo requerido: ~282.0t/h
⚠️ Brecha: -125.0t/h
🔮 Proyección cierre (ritmo actual): ~0.0tn
📌 Déficit estimado al cierre: ~-3000.0tn
─────────────────────
✅ ¿QUÉ SE NECESITA?
─────────────────────
▶️ Horas útiles restantes: 13.38h
▶️ Ritmo mínimo requerido: ~282.0t/h
▶️ Con 60 tn/viaje 
→ sostener 50.0viajes/h

